# VALANCE Metrics

### CATNAP 396 strains

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics as sk
import warnings
import re
warnings.filterwarnings('ignore')

# Define paths
GROUND_TRUTH_FILE = "/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/geomean_labels_399_more_to_add_oct1_25_removed_bad_pub_records_include_TBDs_GTlabels_flipped.txt"


# Specify the threshold (0.2, 1.0, or 50.0)
# THRESHOLD = 50.0  # Change this to 0.2, 1.0, or 50.0
# THRESHOLD = 1.0
THRESHOLD = 0.2

# List of antibodies to process
ANTIBODIES = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'b12', 'VRC03', '2F5']  


# Base directory for prediction files
PREDICTION_BASE_DIR = "/home/yujieq/work/ML_training/VALANCE"

# Output directory (will be created if doesn't exist)
OUTPUT_DIR = "/home/yujieq/work/ML_training/VALANCE/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output Excel file
OUTPUT_EXCEL = os.path.join(OUTPUT_DIR, f"metrics_thres{THRESHOLD}.xlsx")
# ===================================

def extract_virus_id(virus_str):
    if pd.isna(virus_str):
        return None
    parts = str(virus_str).split('.')
    if len(parts) >= 3:
        return parts[-2]
    return str(virus_str)

def extract_resistant_prob(prob_str):
    """Extract the second probability (resistant class) from string like '[0.8808056  0.11919439]'"""
    if pd.isna(prob_str):
        return np.nan
    
    # Convert to string and extract numbers
    prob_str = str(prob_str)
    
    # Find all float numbers in the string
    numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', prob_str)
    
    if len(numbers) >= 2:
        # Return the second number (resistant probability)
        return float(numbers[1])
    elif len(numbers) == 1:
        # If only one number, assume it's the resistant probability
        return float(numbers[0])
    else:
        return np.nan

# Read ground truth
gt_df = pd.read_csv(GROUND_TRUTH_FILE, sep='\t')
gt_df = gt_df[gt_df['Threshold'] == THRESHOLD]
gt_df = gt_df[~gt_df['GeoMean_IC50'].isna()]
gt_df['Virus_clean'] = gt_df['Virus'].apply(extract_virus_id)
gt_df = gt_df.dropna(subset=['Virus_clean', 'Label_0/1'])
gt_df['Label_0/1'] = gt_df['Label_0/1'].astype(int)

# Remove duplicates
gt_df = gt_df.drop_duplicates(subset=['Antibody', 'Virus_clean'])

results = []
all_predictions = []

for antibody in ANTIBODIES:
    
    # Construct prediction file path
    # pred_file = os.path.join(PREDICTION_BASE_DIR, antibody, f"{antibody}_prediction_thres{THRESHOLD}.txt")
    pred_file = os.path.join(PREDICTION_BASE_DIR, antibody, f"{antibody}_prediction_thres{THRESHOLD}_finetuned_tabpfn.txt")
    # pred_file = os.path.join(PREDICTION_BASE_DIR, antibody, f"{antibody}_prediction_thres{THRESHOLD}_autotabpfn.txt")
    
    
    print(f"Processing {antibody} from {pred_file}")
    
    # Read prediction file (no header, columns: virus, prob, prediction)
    try:
        pred_df = pd.read_csv(pred_file, sep='\t', header=None, 
                              names=['virus_id', 'probability', 'prediction'])
    except Exception as e:
        print(f"Error reading {pred_file}: {e}")
        continue
    
    # Extract resistant probability from the probability column
    pred_df['prob_resistant'] = pred_df['probability'].apply(extract_resistant_prob)
    
    # Clean virus IDs
    pred_df['virus_id_clean'] = pred_df['virus_id'].apply(extract_virus_id)
    
    # Filter for specific antibody in ground truth
    antibody_gt = gt_df[gt_df['Antibody'] == antibody].copy()
    
    if antibody_gt.empty:
        print(f"No GT for: {antibody}")
        continue
    
    predictions = []
    ground_truth = []
    probabilities = []  # for AUC
    
    for _, row in antibody_gt.iterrows():
        virus_id = row['Virus_clean']
        gt_label = row['Label_0/1']
        
        pred_rows = pred_df[pred_df['virus_id_clean'] == virus_id]
        
        if not pred_rows.empty:
            # Get prediction from 3rd column
            pred_value = pred_rows['prediction'].iloc[0]
            
            # Convert prediction to integer
            if isinstance(pred_value, str):
                pred_label = 0 if pred_value.lower() == 'sensitive' else 1
            else:
                pred_label = int(pred_value)
            
            # Get resistant probability (already extracted)
            prob_resistant = pred_rows['prob_resistant'].iloc[0]
            
            all_predictions.append({
                'Antibody': antibody,
                'Virus': virus_id,
                'Threshold': THRESHOLD,
                'BRAVE Prediction': pred_label,
                'GT Label': gt_label,
                'Raw Prediction': pred_value,
                'Raw Probability': pred_rows['probability'].iloc[0],
                'Prob_Resistant': prob_resistant
            })
            
            predictions.append(pred_label)
            ground_truth.append(gt_label)
            probabilities.append(prob_resistant)
    
    if not predictions:
        print(f"No matches for: {antibody}")
        continue
    
    # Calculate metrics
    acc = sk.accuracy_score(ground_truth, predictions)
    mcc = sk.matthews_corrcoef(ground_truth, predictions)
    
    # AUC using probability of resistant
    try:
        auc = sk.roc_auc_score(ground_truth, probabilities)
    except ValueError:
        auc = np.nan
    
    precision = sk.precision_score(ground_truth, predictions, zero_division=0)
    recall = sk.recall_score(ground_truth, predictions, zero_division=0)
    f1 = sk.f1_score(ground_truth, predictions, zero_division=0)
    
    cm = sk.confusion_matrix(ground_truth, predictions)
    
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    elif cm.size == 1:
        if predictions[0] == 0:
            tn, fp, fn, tp = cm[0,0], 0, 0, 0
        else:
            tn, fp, fn, tp = 0, 0, 0, cm[0,0]
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    
    results.append({
        'Antibody': antibody,
        'Threshold': THRESHOLD,
        'Num_Matches': len(predictions),
        'MCC': mcc,
        'Accuracy': acc,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'True_Positive': tp,
        'False_Positive': fp,
        'True_Negative': tn,
        'False_Negative': fn
    })
    
    print(f"  Processed {antibody}: {len(predictions)} matches, AUC={auc:.3f}, ACC={acc:.3f}")

# Save results
if results:
    results_df = pd.DataFrame(results)
    all_preds_df = pd.DataFrame(all_predictions)
    
    # Convert to float for better Excel formatting
    float_cols = ['Accuracy', 'AUC', 'MCC', 'Precision', 'Recall', 'F1']
    results_df[float_cols] = results_df[float_cols].astype(float)
    
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='performance_metrics', index=False)
        all_preds_df.to_excel(writer, sheet_name='all_predictions_vs_gt', index=False)
    
    print(f"\nMetrics saved to: {OUTPUT_EXCEL}")
    print(f"\nResults for {len(results_df)} antibodies at threshold {THRESHOLD}:")
    print(results_df.to_string(index=False))
else:
    print("No results were generated. Please check file paths and configurations.")

Processing PGT121 from /home/yujieq/work/ML_training/VALANCE/PGT121/PGT121_prediction_thres0.2_finetuned_tabpfn.txt
  Processed PGT121: 196 matches, AUC=0.814, ACC=0.689
Processing VRC01 from /home/yujieq/work/ML_training/VALANCE/VRC01/VRC01_prediction_thres0.2_finetuned_tabpfn.txt
  Processed VRC01: 270 matches, AUC=0.639, ACC=0.778
Processing 10-1074 from /home/yujieq/work/ML_training/VALANCE/10-1074/10-1074_prediction_thres0.2_finetuned_tabpfn.txt
  Processed 10-1074: 157 matches, AUC=0.747, ACC=0.739
Processing PGT145 from /home/yujieq/work/ML_training/VALANCE/PGT145/PGT145_prediction_thres0.2_finetuned_tabpfn.txt
  Processed PGT145: 199 matches, AUC=0.801, ACC=0.709
Processing 3BNC117 from /home/yujieq/work/ML_training/VALANCE/3BNC117/3BNC117_prediction_thres0.2_finetuned_tabpfn.txt
  Processed 3BNC117: 246 matches, AUC=0.715, ACC=0.598
Processing PGDM1400 from /home/yujieq/work/ML_training/VALANCE/PGDM1400/PGDM1400_prediction_thres0.2_finetuned_tabpfn.txt
  Processed PGDM1400: 12

### Xueling W. 11 test strains

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics as sk
import warnings
import re
warnings.filterwarnings('ignore')


GROUND_TRUTH_FILE = "/home/yujieq/work/ML_training/VALANCE/scripts/geomean_labels_xueling_11test_strain.txt"
PREDICTION_BASE_DIR = "/home/yujieq/work/ML_training/VALANCE"
OUTPUT_DIR  = "/home/yujieq/work/ML_training/VALANCE/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

THRESHOLD  = 1.0   # change to 0.2, 1.0, or 50.0
ANTIBODIES = ['PGT121', 'VRC01', '10-1074', 'N6', '3BNC117', 'PGDM1400', 'SF12', 'PGT145']

OUTPUT_EXCEL = os.path.join(OUTPUT_DIR, f"metrics_xueling_thres{THRESHOLD}.xlsx")


# Manual mapping: GT virus name (as it appears in ground truth file)
#               -> virus ID as it appears in prediction file
VIRUS_NAME_MAP = {
    "AD355_m2_1 (B)":              "AD355_m2_1",
    "AD358_m2_1 (B)":              "AD358_m2_1",
    "AD360_m6_22 (B)":             "AD360_m6_22_",
    "AD414_m9_14 (B)":             "AD414_m9_14",
    "AD415_m9_14 (B)":             "AD415_m9_14_",
    "V704_1109 (BF1)":             "V704_1109_140_RE_CS",
    "CRF02_AG_271 (CRF02_AG) XW":  "02_AG.CM.2004.271.EU513197",
    "AD17 (B) XW":                 "B.US.1999.AD17.GU331254",
    "V703_0455_160":               "C.ZA.2019.V703_0455_160_RE_CS.ON890960",
    "98050":                       "B.US.2020.98050_REBOUND_VA0_E1.PP960923",
    "1209_BM_A5":                  "C.MW.2008.1209_BM_A5.HM070562",
}

def extract_virus_id(virus_str):
    """Map GT virus name to prediction file virus ID using the lookup dict."""
    if pd.isna(virus_str):
        return None
    return VIRUS_NAME_MAP.get(str(virus_str).strip(), str(virus_str).strip())

def extract_resistant_prob(prob_str):
    """Extract second probability (resistant class) from '[0.88  0.12]'."""
    if pd.isna(prob_str):
        return np.nan
    numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', str(prob_str))
    if len(numbers) >= 2:
        return float(numbers[1])
    elif len(numbers) == 1:
        return float(numbers[0])
    return np.nan

# Load ground truth 
gt_df = pd.read_csv(GROUND_TRUTH_FILE, sep='\t')
gt_df = gt_df[gt_df['Threshold'] == THRESHOLD]
gt_df = gt_df[~gt_df['GeoMean_IC50'].isna()]
gt_df['Virus_clean'] = gt_df['Virus'].apply(extract_virus_id)
gt_df = gt_df.dropna(subset=['Virus_clean', 'Label_0/1'])
gt_df['Label_0/1'] = gt_df['Label_0/1'].astype(int)
gt_df = gt_df.drop_duplicates(subset=['Antibody', 'Virus_clean'])

# Collect predictions across all antibodies
all_predictions = []

for antibody in ANTIBODIES:
    pred_file = os.path.join(
        PREDICTION_BASE_DIR, antibody,
        f"{antibody}_prediction_thres{THRESHOLD}_finetuned_tabpfn_xueling_11test_strains.txt"
    )

    if not os.path.exists(pred_file):
        print(f"[SKIP] File not found: {pred_file}")
        continue

    print(f"Processing {antibody} ...")

    try:
        pred_df = pd.read_csv(pred_file, sep='\t', header=None,
                              names=['virus_id', 'probability', 'prediction'])
    except Exception as e:
        print(f"  Error reading {pred_file}: {e}")
        continue

    pred_df['prob_resistant'] = pred_df['probability'].apply(extract_resistant_prob)

    antibody_gt = gt_df[gt_df['Antibody'] == antibody].copy()
    if antibody_gt.empty:
        print(f"  No ground truth found for {antibody}")
        continue

    matched = 0
    for _, row in antibody_gt.iterrows():
        virus_id = row['Virus_clean']
        gt_label  = row['Label_0/1']
        raw_ic50  = row.get('GeoMean_IC50', np.nan)

        pred_rows = pred_df[pred_df['virus_id'] == virus_id]
        if pred_rows.empty:
            print(f"  [UNMATCHED] {antibody} / GT='{row['Virus']}' -> mapped='{virus_id}'")
            continue

        pred_value = pred_rows['prediction'].iloc[0]
        if isinstance(pred_value, str):
            pred_label = 0 if pred_value.lower() == 'sensitive' else 1
        else:
            pred_label = int(pred_value)

        prob_resistant = pred_rows['prob_resistant'].iloc[0]

        all_predictions.append({
            'Antibody':        antibody,
            'Virus':           virus_id,
            'Threshold':       THRESHOLD,
            'GeoMean_IC50':    raw_ic50,
            'GT Label':        gt_label,
            'Prediction':      pred_label,
            'Prob_Resistant':  prob_resistant,
            'Raw Prediction':  pred_value,
            'Raw Probability': pred_rows['probability'].iloc[0],
        })
        matched += 1

    print(f"  {matched} matched strains")

# Pooled metrics
if not all_predictions:
    print("No results generated. Check file paths.")
else:
    all_preds_df = pd.DataFrame(all_predictions)

    y_true = all_preds_df["GT Label"].values
    y_pred = all_preds_df["Prediction"].values
    y_prob = all_preds_df["Prob_Resistant"].values

    try:
        auc = sk.roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    pooled = {
        "Threshold":  THRESHOLD,
        "N_total":    len(y_true),
        "N_antibodies": len(all_preds_df["Antibody"].unique()),
        "MCC":        sk.matthews_corrcoef(y_true, y_pred),
        "Accuracy":   sk.accuracy_score(y_true, y_pred),
        "AUC":        auc,
        "F1":         sk.f1_score(y_true, y_pred, zero_division=0),
    }

    print(f"\n{'='*55}")
    print(f"Pooled metrics — threshold {THRESHOLD} µg/mL")
    print(f"  {pooled['N_total']} predictions  "
          f"({pooled['N_antibodies']} antibodies × ~10 strains)")
    print(f"{'='*55}")
    for k, v in pooled.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    pooled_df = pd.DataFrame([pooled])
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        pooled_df.to_excel(writer,    sheet_name='pooled_metrics',         index=False)
        all_preds_df.to_excel(writer, sheet_name='all_predictions_vs_gt',  index=False)

    print(f"\n✓ Saved to: {OUTPUT_EXCEL}")

Processing PGT121 ...
  11 matched strains
Processing VRC01 ...
  [UNMATCHED] VRC01 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing 10-1074 ...
  11 matched strains
Processing N6 ...
  11 matched strains
Processing 3BNC117 ...
  11 matched strains
Processing PGDM1400 ...
  [UNMATCHED] PGDM1400 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing SF12 ...
  11 matched strains
Processing PGT145 ...
  11 matched strains

Pooled metrics — threshold 1.0 µg/mL
  86 predictions  (8 antibodies × ~10 strains)
  Threshold: 1.0000
  N_total: 86
  N_antibodies: 8
  MCC: 0.6175
  Accuracy: 0.7907
  AUC: 0.8620
  F1: 0.7568

✓ Saved to: /home/yujieq/work/ML_training/VALANCE/metrics/metrics_xueling_thres1.0.xlsx


# bNab-rep metrics

### CATNAP 396 strains

In [9]:
import pandas as pd
import sklearn.metrics as sk
from pathlib import Path

# ------------------------------------------------------------------------------
# 1) Paths and parameters
# ------------------------------------------------------------------------------
# Specify which threshold to use (0.2, 1.0, or 50.0)
# thres = 0.2  
# thres = 1.0
thres = 50.0 

base_pred_dir = Path(f"/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_{thres}")
geo_txt = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/geomean_labels_399_more_to_add_oct1_25_removed_bad_pub_records_include_TBDs_GTlabels_flipped.txt")

antibodies = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'b12', 'VRC03', '2F5']

output_xlsx = base_pred_dir / f"bNAb_ReP_metrics_thres{thres}.xlsx"

# ------------------------------------------------------------------------------
# 2) Load GT data
# ------------------------------------------------------------------------------
geo_df = pd.read_csv(geo_txt, sep='\t', dtype=str)
geo_df = geo_df.rename(columns={'Label_Sensitive/Resistant': 'GT Label'})

# Keep only rows with the specified threshold
geo_df = geo_df[geo_df['Threshold'].astype(float) == thres]

# Keep only needed columns
geo_df = geo_df[['Antibody', 'Virus', 'GT Label']]

print(f"Using threshold = {thres}")
print(f"Total GT entries for threshold {thres}: {len(geo_df)}")

# ------------------------------------------------------------------------------
# 3) Process
# ------------------------------------------------------------------------------
all_pred_vs_gt = []
all_metrics = []
summary_stats = []

for antibody in antibodies:
    print(f"\nProcessing antibody: {antibody}")

    pred_file = base_pred_dir / antibody / "predictions" / "396_strains_probabilities.csv"

    if not pred_file.exists():
        print(f"Missing file: {pred_file}")
        continue

    df = pd.read_csv(pred_file)

    records = []
    for _, row in df.iterrows():
        virus_id = str(row['id']).split('.')[-2]
        prob = float(row['probability'])
        pred_label = 'sensitive' if prob < 0.5 else 'resistant'

        records.append({
            'Antibody': antibody,
            'Virus': virus_id,
            'Prob': prob,
            'Predicted Label': pred_label
        })

    pred_df = pd.DataFrame(records)

    # Merge with GT (now filtered by threshold)
    merged = pred_df.merge(
        geo_df,
        on=['Antibody', 'Virus'],
        how='inner'
    )

    print(f"Total matched (with GT for threshold {thres}): {len(merged)}")

    # Binary encoding (correct)
    merged['GT_bin'] = merged['GT Label'].map({'sensitive': 0, 'resistant': 1}).astype(int)
    merged['Pred_bin'] = merged['Predicted Label'].map({'sensitive': 0, 'resistant': 1}).astype(int)

    gt = merged['GT_bin'].values
    pred = merged['Pred_bin'].values
    prob = merged['Prob'].values

    # Metrics
    acc = sk.accuracy_score(gt, pred)
    mcc = sk.matthews_corrcoef(gt, pred)
    f1 = sk.f1_score(gt, pred, zero_division=0)
    try:
        auc = sk.roc_auc_score(gt, prob)
    except ValueError:
        auc = float('nan')

    all_metrics.append({
        'Antibody': antibody,
        'MCC': mcc,
        'AUC': auc,
        'ACC': acc,
        'F1': f1,
        'N': len(gt)
    })

    all_pred_vs_gt.append(merged)

    summary_stats.append({
        'Antibody': antibody,
        'Total_Instances': len(merged)
    })

# ------------------------------------------------------------------------------
# 4) Combine
# ------------------------------------------------------------------------------
pred_vs_gt_df = pd.concat(all_pred_vs_gt, ignore_index=True) if all_pred_vs_gt else pd.DataFrame()
metrics_df = pd.DataFrame(all_metrics) if all_metrics else pd.DataFrame()
summary_df = pd.DataFrame(summary_stats) if summary_stats else pd.DataFrame()

# ------------------------------------------------------------------------------
# 5) Save Excel (1 scenario + summary)
# ------------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"Saving results to Excel for threshold = {thres}...")
print(f"{'='*60}")

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:

    # Tab 1: pred_vs_gt
    if not pred_vs_gt_df.empty:
        pred_vs_gt_df.to_excel(writer, sheet_name="pred_vs_gt", index=False)
    else:
        pd.DataFrame({"note": ["No pred_vs_gt data"]}).to_excel(
            writer, sheet_name="pred_vs_gt", index=False
        )

    # Tab 2: metrics + summary
    summary_sheet = "per_ab_metrics_summary"
    startrow = 0

    if not metrics_df.empty:
        # Add note about which threshold was used
        pd.DataFrame({"threshold_used": [thres]}).to_excel(
            writer, sheet_name=summary_sheet, index=False, startrow=startrow
        )
        startrow += 3
        
        metrics_df.to_excel(writer, sheet_name=summary_sheet, index=False, startrow=startrow)
        startrow += len(metrics_df) + 3
    else:
        pd.DataFrame({"note": ["No metrics data"]}).to_excel(
            writer, sheet_name=summary_sheet, index=False, startrow=startrow
        )
        startrow += 5

    if not summary_df.empty:
        summary_df.to_excel(writer, sheet_name=summary_sheet, index=False, startrow=startrow)
    else:
        pd.DataFrame({"note": ["No summary data"]}).to_excel(
            writer, sheet_name=summary_sheet, index=False, startrow=startrow
        )

print(f"\nSaved Excel to: {output_xlsx}")

Using threshold = 50.0
Total GT entries for threshold 50.0: 3060

Processing antibody: PGT121
Total matched (with GT for threshold 50.0): 196

Processing antibody: VRC01
Total matched (with GT for threshold 50.0): 270

Processing antibody: 10-1074
Total matched (with GT for threshold 50.0): 157

Processing antibody: PGT145
Total matched (with GT for threshold 50.0): 199

Processing antibody: 3BNC117
Total matched (with GT for threshold 50.0): 246

Processing antibody: PGDM1400
Total matched (with GT for threshold 50.0): 123

Processing antibody: VRC26.25
Total matched (with GT for threshold 50.0): 41

Processing antibody: PGT151
Total matched (with GT for threshold 50.0): 124

Processing antibody: PG9
Total matched (with GT for threshold 50.0): 206

Processing antibody: 4E10
Total matched (with GT for threshold 50.0): 152

Processing antibody: PGT128
Total matched (with GT for threshold 50.0): 175

Processing antibody: SF12
Total matched (with GT for threshold 50.0): 35

Processing ant

### Xueling W. 11 test strains

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics as sk
import warnings
warnings.filterwarnings('ignore')

GROUND_TRUTH_FILE = "/home/yujieq/work/ML_training/VALANCE/scripts/geomean_labels_xueling_11test_strain.txt"
BNAB_REP_BASE_DIR = "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv"
OUTPUT_DIR = "/home/yujieq/work/ML_training/VALANCE/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ANTIBODIES = ['PGT121', 'VRC01', '10-1074', 'N6', '3BNC117', 'PGDM1400', 'SF12', 'PGT145']


THRESHOLDS = [
    {"gt_value": 0.2,  "folder": "0.2"},
    {"gt_value": 1.0,  "folder": "1"},
    {"gt_value": 50.0, "folder": "50"},
]

OUTPUT_EXCEL = os.path.join(OUTPUT_DIR, "metrics_xueling_bNAbReP_all_thresholds.xlsx")


# Manual mapping: GT virus name (as it appears in ground truth file)
#               -> virus ID as it appears in prediction file
VIRUS_NAME_MAP = {
    "AD355_m2_1 (B)":              "AD355_m2_1",
    "AD358_m2_1 (B)":              "AD358_m2_1",
    "AD360_m6_22 (B)":             "AD360_m6_22_",
    "AD414_m9_14 (B)":             "AD414_m9_14",
    "AD415_m9_14 (B)":             "AD415_m9_14_",
    "V704_1109 (BF1)":             "V704_1109_140_RE_CS",
    "CRF02_AG_271 (CRF02_AG) XW":  "02_AG.CM.2004.271.EU513197",
    "AD17 (B) XW":                 "B.US.1999.AD17.GU331254",
    "V703_0455_160":               "C.ZA.2019.V703_0455_160_RE_CS.ON890960",
    "98050":                       "B.US.2020.98050_REBOUND_VA0_E1.PP960923",
    "1209_BM_A5":                  "C.MW.2008.1209_BM_A5.HM070562",
}

def extract_virus_id(virus_str):
    """Map GT virus name to prediction file virus ID using the lookup dict."""
    if pd.isna(virus_str):
        return None
    return VIRUS_NAME_MAP.get(str(virus_str).strip(), str(virus_str).strip())

def build_prediction_path(antibody, folder):
    return os.path.join(
        BNAB_REP_BASE_DIR,
        f"IC50_{folder}",
        antibody,
        "predictions",
        "xueling_11test_probabilities.csv"
    )

def compute_pooled_metrics(gt_df_all, threshold_gt_value, threshold_folder):
    """Build the matched prediction/ground-truth table and pooled metrics
    for one threshold, across all 8 antibodies."""
    gt_df = gt_df_all[gt_df_all['Threshold'] == threshold_gt_value]
    gt_df = gt_df[~gt_df['GeoMean_IC50'].isna()]
    gt_df = gt_df.copy()
    gt_df['Virus_clean'] = gt_df['Virus'].apply(extract_virus_id)
    gt_df = gt_df.dropna(subset=['Virus_clean', 'Label_0/1'])
    gt_df['Label_0/1'] = gt_df['Label_0/1'].astype(int)
    gt_df = gt_df.drop_duplicates(subset=['Antibody', 'Virus_clean'])

    all_predictions = []

    for antibody in ANTIBODIES:
        pred_file = build_prediction_path(antibody, threshold_folder)

        if not os.path.exists(pred_file):
            print(f"[SKIP] {antibody} @ IC50_{threshold_folder}: file not found: {pred_file}")
            continue

        print(f"Processing {antibody} @ IC50_{threshold_folder} ...")

        try:
            pred_df = pd.read_csv(pred_file)
        except Exception as e:
            print(f"  Error reading {pred_file}: {e}")
            continue

        expected_cols = {'id', 'probability'}
        if not expected_cols.issubset(pred_df.columns):
            print(f"  [WARNING] {pred_file} missing expected columns "
                  f"{expected_cols - set(pred_df.columns)}, skipping.")
            continue

        antibody_gt = gt_df[gt_df['Antibody'] == antibody].copy()
        if antibody_gt.empty:
            print(f"  No ground truth found for {antibody}")
            continue

        matched = 0
        for _, row in antibody_gt.iterrows():
            virus_id = row['Virus_clean']
            gt_label = row['Label_0/1']
            raw_ic50 = row.get('GeoMean_IC50', np.nan)

            pred_rows = pred_df[pred_df['id'] == virus_id]
            if pred_rows.empty:
                print(f"  [UNMATCHED] {antibody} / GT='{row['Virus']}' -> mapped='{virus_id}'")
                continue

            prob_resistant = pred_rows['probability'].iloc[0]
            # pred_label rule: sensitive if prob < 0.5, else resistant
            pred_label = 0 if prob_resistant < 0.5 else 1

            all_predictions.append({
                'Antibody':        antibody,
                'Virus':           virus_id,
                'Threshold':       threshold_folder,
                'GeoMean_IC50':    raw_ic50,
                'GT Label':        gt_label,
                'Prediction':      pred_label,
                'Prob_Resistant':  prob_resistant,
            })
            matched += 1

        print(f"  {matched} matched strains")

    if not all_predictions:
        print(f"No results generated for threshold {threshold_folder}. Check file paths.")
        return None, None

    all_preds_df = pd.DataFrame(all_predictions)

    y_true = all_preds_df["GT Label"].values
    y_pred = all_preds_df["Prediction"].values
    y_prob = all_preds_df["Prob_Resistant"].values

    try:
        auc = sk.roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    pooled = {
        "Threshold":    threshold_folder,
        "N_total":      len(y_true),
        "N_antibodies": len(all_preds_df["Antibody"].unique()),
        "MCC":          sk.matthews_corrcoef(y_true, y_pred),
        "Accuracy":     sk.accuracy_score(y_true, y_pred),
        "AUC":          auc,
        "F1":           sk.f1_score(y_true, y_pred, zero_division=0),
    }

    print(f"\n{'='*55}")
    print(f"Pooled metrics — IC50_{threshold_folder} µg/mL")
    print(f"  {pooled['N_total']} predictions "
          f"({pooled['N_antibodies']} antibodies)")
    print(f"{'='*55}")
    for k, v in pooled.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print()

    return pooled, all_preds_df

# Load ground truth 
gt_df_all = pd.read_csv(GROUND_TRUTH_FILE, sep='\t')

# Run for all thresholds 
summary_rows = []
per_threshold_preds = {}

for spec in THRESHOLDS:
    pooled, all_preds_df = compute_pooled_metrics(gt_df_all, spec["gt_value"], spec["folder"])
    if pooled is not None:
        summary_rows.append(pooled)
        per_threshold_preds[spec["folder"]] = all_preds_df

if not summary_rows:
    print("No results generated for any threshold. Check file paths.")
else:
    summary_df = pd.DataFrame(summary_rows)
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='summary_all_thresholds', index=False)
        for threshold_folder, preds_df in per_threshold_preds.items():
            sheet_name = f"preds_thres{threshold_folder}"[:31]  # Excel sheet name limit
            preds_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n✓ Saved to: {OUTPUT_EXCEL}")

Processing PGT121 @ IC50_0.2 ...
  11 matched strains
Processing VRC01 @ IC50_0.2 ...
  [UNMATCHED] VRC01 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing 10-1074 @ IC50_0.2 ...
  11 matched strains
Processing N6 @ IC50_0.2 ...
  11 matched strains
Processing 3BNC117 @ IC50_0.2 ...
  11 matched strains
Processing PGDM1400 @ IC50_0.2 ...
  [UNMATCHED] PGDM1400 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing SF12 @ IC50_0.2 ...
  11 matched strains
Processing PGT145 @ IC50_0.2 ...
  11 matched strains

Pooled metrics — IC50_0.2 µg/mL
  86 predictions (8 antibodies)
  Threshold: 0.2
  N_total: 86
  N_antibodies: 8
  MCC: 0.3118
  Accuracy: 0.6628
  AUC: 0.7645
  F1: 0.7434

Processing PGT121 @ IC50_1 ...
  11 matched strains
Processing VRC01 @ IC50_1 ...
  [UNMATCHED] VRC01 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing 10-1074 @ IC50_1 ...
  11 matched strains
Processing N

# BRAVE metrics

### CATNAP 396 strains

In [9]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics as sk
import warnings
warnings.filterwarnings('ignore')

# Define paths
BRAVE_DIR = "/home/yujieq/work/ML_training/BRAVE/"
GROUND_TRUTH_FILE = "/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/geomean_labels_399_more_to_add_oct1_25_removed_bad_pub_records_include_TBDs_GTlabels_flipped.txt"
OUTPUT_EXCEL = "/home/yujieq/work/ML_training/BRAVE/BRAVE_metrics.xlsx"

ANTIBODIES = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 
    'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', 
    '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 
    'b12', 'VRC03', '2F5']

def extract_virus_id(virus_str):
    if pd.isna(virus_str):
        return None
    parts = str(virus_str).split('.')
    if len(parts) >= 3:
        return parts[-2]
    return str(virus_str)

# Read ground truth
gt_df = pd.read_csv(GROUND_TRUTH_FILE, sep='\t')
gt_df = gt_df[gt_df['Threshold'] == 0.2]
gt_df = gt_df[~gt_df['GeoMean_IC50'].isna()]
gt_df['Virus_clean'] = gt_df['Virus'].apply(extract_virus_id)
gt_df = gt_df.dropna(subset=['Virus_clean', 'Label_0/1'])
gt_df['Label_0/1'] = gt_df['Label_0/1'].astype(int)

# # ---- Flip labels so resistant=1, sensitive=0 ----
# gt_df['Label_0/1'] = 1 - gt_df['Label_0/1']

gt_df = gt_df.drop_duplicates(subset=['Antibody', 'Virus_clean'])

results = []
all_predictions = []

for antibody in ANTIBODIES:

    pred_file = f"thres_0.2/{antibody}_output_full_CATNAP.csv"
    pred_path = os.path.join(BRAVE_DIR, pred_file)

    if not os.path.exists(pred_path):
        print(f"Missing: {antibody}")
        continue

    pred_df = pd.read_csv(pred_path)
    pred_df['virus_id_clean'] = pred_df['virus_id'].apply(extract_virus_id)

    antibody_gt = gt_df[gt_df['Antibody'] == antibody].copy()

    if antibody_gt.empty:
        print(f"No GT for: {antibody}")
        continue

    predictions = []
    ground_truth = []
    probabilities = []   # <-- for AUC

    for _, row in antibody_gt.iterrows():

        virus_id = row['Virus_clean']
        gt_label = row['Label_0/1']

        pred_rows = pred_df[pred_df['virus_id_clean'] == virus_id]

        if not pred_rows.empty and 'prediction' in pred_rows.columns:

            pred_value = pred_rows['prediction'].iloc[0]

            # ---- flip prediction labels: resistant=1, sensitive=0 ----
            if isinstance(pred_value, str):
                pred_label = 0 if pred_value.lower() == 'sensitive' else 1
            else:
                pred_label = int(pred_value)

            # ---- get probability of resistant ----
            prob_resistant = pred_rows['probability.Resistant'].iloc[0]

            all_predictions.append({
                'Antibody': antibody,
                'Virus': virus_id,
                'Threshold': 50.0,
                'BRAVE Prediction': pred_label,
                'GT Label': gt_label,
                'Raw Prediction': pred_value,
                'Prob_Resistant': prob_resistant
            })

            predictions.append(pred_label)
            ground_truth.append(gt_label)
            probabilities.append(prob_resistant)

    if not predictions:
        print(f"No matches for: {antibody}")
        continue

    acc = sk.accuracy_score(ground_truth, predictions)
    mcc = sk.matthews_corrcoef(ground_truth, predictions)

    # ---- AUC now uses probability of resistant ----
    try:
        auc = sk.roc_auc_score(ground_truth, probabilities)
    except ValueError:
        auc = np.nan

    precision = sk.precision_score(ground_truth, predictions, zero_division=0)
    recall = sk.recall_score(ground_truth, predictions, zero_division=0)
    f1 = sk.f1_score(ground_truth, predictions, zero_division=0)

    cm = sk.confusion_matrix(ground_truth, predictions)

    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    elif cm.size == 1:
        if predictions[0] == 0:
            tn, fp, fn, tp = cm[0,0], 0, 0, 0
        else:
            tn, fp, fn, tp = 0, 0, 0, cm[0,0]
    else:
        tn, fp, fn, tp = 0, 0, 0, 0

    results.append({
        'Antibody': antibody,
        'Num_Matches': len(predictions),
        'MCC': mcc,
        'AUC': auc,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'True_Positive': tp,
        'False_Positive': fp,
        'True_Negative': tn,
        'False_Negative': fn
    })

results_df = pd.DataFrame(results)
all_preds_df = pd.DataFrame(all_predictions)

results_df[['Accuracy','AUC','MCC','Precision','Recall','F1']] = results_df[['Accuracy','AUC','MCC','Precision','Recall','F1']].astype(float)

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='performance_metrics', index=False)
    all_preds_df.to_excel(writer, sheet_name='all_antibodies_pred_vs_gt', index=False)

print(f"Metrics saved to: {OUTPUT_EXCEL}")
print(f"\nResults for {len(results_df)} antibodies:")
print(results_df.to_string(index=False))

Metrics saved to: /home/yujieq/work/ML_training/BRAVE/BRAVE_metrics.xlsx

Results for 22 antibodies:
Antibody  Num_Matches       MCC      AUC  Accuracy  Precision   Recall       F1  True_Positive  False_Positive  True_Negative  False_Negative
  PGT121          196  0.000000 0.763793  0.591837   0.591837 1.000000 0.743590            116              80              0               0
   VRC01          270  0.000000 0.485696  0.866667   0.866667 1.000000 0.928571            234              36              0               0
 10-1074          157  0.000000 0.731114  0.471338   0.471338 1.000000 0.640693             74              83              0               0
  PGT145          199  0.226321 0.676046  0.603015   0.580460 0.943925 0.718861            101              73             19               6
 3BNC117          246  0.259129 0.728898  0.589431   0.823529 0.227642 0.356688             28               6            117              95
PGDM1400          123  0.253289 0.751030  0.764

## Nested CV from averaging 5 outer folds

cd /data/BRAVE/Training/final_models/thres_50/

R

load("35O22_full_CATNAP.RData")

library(pROC)

library(mltools)

metrics_list <- lapply(1:5, function(i) {
    fold_preds <- res$outer_result[[i]]$preds
    
    obs  <- fold_preds$testy
    prob <- fold_preds$predyp
    pred <- fold_preds$predy
    
    # Confusion matrix components
    tp <- sum(pred == "Resistant" & obs == "Resistant")
    tn <- sum(pred == "Sensitive" & obs == "Sensitive")
    fp <- sum(pred == "Resistant" & obs == "Sensitive")
    fn <- sum(pred == "Sensitive" & obs == "Resistant")
    
    # AUC
    roc_obj <- roc(obs, prob, levels = c("Sensitive", "Resistant"), quiet = TRUE)
    fold_auc <- as.numeric(auc(roc_obj))
    
    # Accuracy
    fold_acc <- mean(obs == pred)
    
    # MCC
    fold_mcc <- mcc(TP = tp, TN = tn, FP = fp, FN = fn)
    
    # F1
    precision <- tp / (tp + fp)
    recall    <- tp / (tp + fn)
    fold_f1   <- 2 * (precision * recall) / (precision + recall)
    
    c(AUC = fold_auc, ACC = fold_acc, MCC = fold_mcc, F1 = fold_f1)
})

metrics_df <- do.call(rbind, metrics_list)
rownames(metrics_df) <- paste0("Fold", 1:5)

print(metrics_df)
cat("\nMean:\n"); print(colMeans(metrics_df))
cat("\nSD:\n");   print(apply(metrics_df, 2, sd))

### Xueling W. 11 test strains

In [2]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn import metrics as sk
import warnings
import re
warnings.filterwarnings('ignore')


GROUND_TRUTH_FILE   = "/home/yujieq/work/ML_training/VALANCE/scripts/geomean_labels_xueling_11test_strain.txt"
BRAVE_PREDICTION_DIR = "/home/yujieq/work/ML_training/BRAVE"
OUTPUT_DIR  = "/home/yujieq/work/ML_training/VALANCE/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ANTIBODIES = ['PGT121', 'VRC01', '10-1074', 'N6', '3BNC117', 'PGDM1400', 'SF12', 'PGT145']


THRESHOLDS = [
    {"gt_value": 0.2,  "candidates": ["0.2", "0.20"]},
    {"gt_value": 1.0,  "candidates": ["1", "1.0"]},
    {"gt_value": 50.0, "candidates": ["50", "50.0"]},
]

OUTPUT_EXCEL = os.path.join(OUTPUT_DIR, "metrics_xueling_BRAVE_all_thresholds.xlsx")


# Manual mapping: GT virus name (as it appears in ground truth file)
#               -> virus ID as it appears in prediction file
VIRUS_NAME_MAP = {
    "AD355_m2_1 (B)":              "AD355_m2_1",
    "AD358_m2_1 (B)":              "AD358_m2_1",
    "AD360_m6_22 (B)":             "AD360_m6_22_",
    "AD414_m9_14 (B)":             "AD414_m9_14",
    "AD415_m9_14 (B)":             "AD415_m9_14_",
    "V704_1109 (BF1)":             "V704_1109_140_RE_CS",
    "CRF02_AG_271 (CRF02_AG) XW":  "02_AG.CM.2004.271.EU513197",
    "AD17 (B) XW":                 "B.US.1999.AD17.GU331254",
    "V703_0455_160":               "C.ZA.2019.V703_0455_160_RE_CS.ON890960",
    "98050":                       "B.US.2020.98050_REBOUND_VA0_E1.PP960923",
    "1209_BM_A5":                  "C.MW.2008.1209_BM_A5.HM070562",
}

def extract_virus_id(virus_str):
    """Map GT virus name to prediction file virus ID using the lookup dict."""
    if pd.isna(virus_str):
        return None
    return VIRUS_NAME_MAP.get(str(virus_str).strip(), str(virus_str).strip())

def find_prediction_file(antibody, candidates):
    tried = []
    for cand in candidates:
        path = os.path.join(
            BRAVE_PREDICTION_DIR,
            f"{antibody}_output_full_CATNAP_Xueling_11tests_thres{cand}.csv"
        )
        tried.append(path)
        if os.path.exists(path):
            return path, tried
    return None, tried

def compute_pooled_metrics(gt_df_all, threshold_gt_value, threshold_label):
    """Build the matched prediction/ground-truth table and pooled metrics
    for one threshold, across all 8 antibodies."""
    gt_df = gt_df_all[gt_df_all['Threshold'] == threshold_gt_value]
    gt_df = gt_df[~gt_df['GeoMean_IC50'].isna()]
    gt_df = gt_df.copy()
    gt_df['Virus_clean'] = gt_df['Virus'].apply(extract_virus_id)
    gt_df = gt_df.dropna(subset=['Virus_clean', 'Label_0/1'])
    gt_df['Label_0/1'] = gt_df['Label_0/1'].astype(int)
    gt_df = gt_df.drop_duplicates(subset=['Antibody', 'Virus_clean'])

    all_predictions = []

    for antibody in ANTIBODIES:
        candidates = [c for c in THRESHOLDS if c["gt_value"] == threshold_gt_value][0]["candidates"]
        pred_file, tried_paths = find_prediction_file(antibody, candidates)

        if pred_file is None:
            print(f"[SKIP] {antibody} @ thres {threshold_label}: no file found. Tried:")
            for p in tried_paths:
                print(f"         {p}")
            continue

        print(f"Processing {antibody} @ thres {threshold_label} ({os.path.basename(pred_file)}) ...")

        try:
            pred_df = pd.read_csv(pred_file)
        except Exception as e:
            print(f"  Error reading {pred_file}: {e}")
            continue

        expected_cols = {'virus_id', 'prediction', 'probability.Resistant'}
        if not expected_cols.issubset(pred_df.columns):
            print(f"  [WARNING] {pred_file} missing expected columns "
                  f"{expected_cols - set(pred_df.columns)}, skipping.")
            continue

        antibody_gt = gt_df[gt_df['Antibody'] == antibody].copy()
        if antibody_gt.empty:
            print(f"  No ground truth found for {antibody}")
            continue

        matched = 0
        for _, row in antibody_gt.iterrows():
            virus_id = row['Virus_clean']
            gt_label = row['Label_0/1']
            raw_ic50 = row.get('GeoMean_IC50', np.nan)

            pred_rows = pred_df[pred_df['virus_id'] == virus_id]
            if pred_rows.empty:
                print(f"  [UNMATCHED] {antibody} / GT='{row['Virus']}' -> mapped='{virus_id}'")
                continue

            raw_prediction = pred_rows['prediction'].iloc[0]
            pred_label = 1 if str(raw_prediction).strip().lower() == 'resistant' else 0
            prob_resistant = pred_rows['probability.Resistant'].iloc[0]

            all_predictions.append({
                'Antibody':        antibody,
                'Virus':           virus_id,
                'Threshold':       threshold_label,
                'GeoMean_IC50':    raw_ic50,
                'GT Label':        gt_label,
                'Prediction':      pred_label,
                'Prob_Resistant':  prob_resistant,
                'Raw Prediction':  raw_prediction,
            })
            matched += 1

        print(f"  {matched} matched strains")

    if not all_predictions:
        print(f"No results generated for threshold {threshold_label}. Check file paths.")
        return None, None

    all_preds_df = pd.DataFrame(all_predictions)

    y_true = all_preds_df["GT Label"].values
    y_pred = all_preds_df["Prediction"].values
    y_prob = all_preds_df["Prob_Resistant"].values

    try:
        auc = sk.roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    pooled = {
        "Threshold":    threshold_label,
        "N_total":      len(y_true),
        "N_antibodies": len(all_preds_df["Antibody"].unique()),
        "MCC":          sk.matthews_corrcoef(y_true, y_pred),
        "Accuracy":     sk.accuracy_score(y_true, y_pred),
        "AUC":          auc,
        "F1":           sk.f1_score(y_true, y_pred, zero_division=0),
    }

    print(f"\n{'='*55}")
    print(f"Pooled metrics — threshold {threshold_label} µg/mL")
    print(f"  {pooled['N_total']} predictions "
          f"({pooled['N_antibodies']} antibodies)")
    print(f"{'='*55}")
    for k, v in pooled.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print()

    return pooled, all_preds_df

# Load ground truth
gt_df_all = pd.read_csv(GROUND_TRUTH_FILE, sep='\t')

# Run for all thresholds
summary_rows = []
per_threshold_preds = {}

for spec in THRESHOLDS:
    threshold_label = spec["candidates"][0]  # e.g. "0.2", "1", "50" - used for display/sheet naming
    pooled, all_preds_df = compute_pooled_metrics(gt_df_all, spec["gt_value"], threshold_label)
    if pooled is not None:
        summary_rows.append(pooled)
        per_threshold_preds[threshold_label] = all_preds_df

if not summary_rows:
    print("No results generated for any threshold. Check file paths.")
else:
    summary_df = pd.DataFrame(summary_rows)
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='summary_all_thresholds', index=False)
        for threshold_label, preds_df in per_threshold_preds.items():
            sheet_name = f"preds_thres{threshold_label}"[:31]  # Excel sheet name limit
            preds_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n✓ Saved to: {OUTPUT_EXCEL}")

Processing PGT121 @ thres 0.2 (PGT121_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  11 matched strains
Processing VRC01 @ thres 0.2 (VRC01_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  [UNMATCHED] VRC01 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing 10-1074 @ thres 0.2 (10-1074_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  11 matched strains
Processing N6 @ thres 0.2 (N6_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  11 matched strains
Processing 3BNC117 @ thres 0.2 (3BNC117_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  11 matched strains
Processing PGDM1400 @ thres 0.2 (PGDM1400_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  [UNMATCHED] PGDM1400 / GT='V704_1109 (BF1)' -> mapped='V704_1109_140_RE_CS'
  10 matched strains
Processing SF12 @ thres 0.2 (SF12_output_full_CATNAP_Xueling_11tests_thres0.2.csv) ...
  11 matched strains
Processing PGT145 @ thres 0.2 (PGT145_output_full_CATNAP_Xueling_11te

# FC-ATT-GRU metrics

In [1]:
import pandas as pd
import sklearn.metrics as sk
from pathlib import Path
import glob
from collections import defaultdict

# ------------------------------------------------------------------------------
# 1) Paths and parameters
# ------------------------------------------------------------------------------
base_pred_dir = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/prediction_results/CATNAP_399_strains/curated_data_epitope_with_all_lineage")
base_out_dir  = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/prediction_results/CATNAP_399_strains/curated_data_epitope_with_all_lineage")


geo_txt_include = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/geomean_labels_399_more_to_add_oct1_25_removed_bad_pub_records_include_TBDs_GTlabels_flipped.txt")


viruses_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/viruses_oct1st2025.txt")
antibodies = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'VRC34.01', 'b12', 'VRC07-523LS.v34', 'VRC03', 'VRC07', '2F5', '8ANC195']
thresholds = [0.2, 1, 50]

EXCLUDED_VIRUSES = {
    "BC11",
    "T250_4",
    "T252_7",
}

# Create a dictionary that maps scenario names to a dictionary of threshold directories
pred_dirs = {
    "include_TBDs": {
        threshold: base_pred_dir / f"threshold_{threshold}" / "include_TBDs_GTlabels_flipped" / "5folds_1_repeat"
        for threshold in thresholds
    }
}

geo_files = {
    "include_TBDs": geo_txt_include
}

# 3 scenario tabs + 1 combined summary tab
output_xlsx = base_out_dir / "curated_data_epitope_with_all_lineage_model_5folds_1_repeat_pred_vs_gt_include_TBDs_GTlabels_flipped.xlsx"

# ------------------------------------------------------------------------------
# 2) Load viruses data to get clade information
# ------------------------------------------------------------------------------
print("Loading viruses file for clade information...")
viruses_df = pd.read_csv(viruses_file, sep='\t', dtype=str)
viruses_clade_df = viruses_df[['---Virus name', 'Subtype']].copy()
viruses_clade_df = viruses_clade_df.rename(columns={'---Virus name': 'Virus', 'Subtype': 'Clade'})
print(f"Loaded clade information for {len(viruses_clade_df)} viruses")

# ------------------------------------------------------------------------------
# helper: load geo labels
# ------------------------------------------------------------------------------
def load_geo_df(geo_txt_path: Path) -> pd.DataFrame:
    geo_df = pd.read_csv(geo_txt_path, sep='\t', dtype=str)
    geo_df = geo_df.rename(columns={'Label_Sensitive/Resistant': 'GT Label'})[
        ['Antibody','Virus','Threshold','GeoMean_IC50','GT Label']
    ]
    geo_df['Threshold'] = pd.to_numeric(geo_df['Threshold'], errors='coerce')
    geo_df['GeoMean_IC50'] = pd.to_numeric(geo_df['GeoMean_IC50'], errors='coerce')
    geo_df = geo_df.dropna(subset=['GeoMean_IC50'])
    return geo_df

# ------------------------------------------------------------------------------
# 3) Run same pipeline for 3 scenarios
# ------------------------------------------------------------------------------
scenario_results = {}  # scenario -> dict(pred_vs_gt_df, metrics_df, summary_df)

# Collect per-antibody metrics + summary across scenarios for the 4th tab
all_scenarios_metrics = []
all_scenarios_summary = []

for scenario_name in ["include_TBDs"]:
    print(f"\n{'='*80}")
    print(f"SCENARIO: {scenario_name}")
    print(f"{'='*80}")

    pred_dir_dict = pred_dirs[scenario_name]  # This is now a dict mapping threshold to directory
    geo_txt = geo_files[scenario_name]

    geo_df = load_geo_df(geo_txt)

    all_pred_vs_gt_data = []
    all_metrics_data = []
    summary_stats = []

    for antibody in antibodies:
        print(f"\n{'='*60}")
        print(f"Processing antibody: {antibody}")
        print(f"{'='*60}")

        pred_frames = []
        for thr in thresholds:
            pred_dir = pred_dir_dict[thr]  # Get the directory for this threshold
            if not pred_dir.exists():
                print(f"  Missing prediction directory: {pred_dir}")
                continue

            expected_count = 5
            pattern = f"Predicted_threshold{thr}_CATNAP_399_more_to_add_oct1_25_fold*_repeat1_{antibody}.txt"
            pred_files = glob.glob(str(pred_dir / pattern))

            if len(pred_files) < expected_count:
                print(f"  Found only {len(pred_files)} files for {antibody} threshold {thr}, expected {expected_count}")

            virus_predictions = defaultdict(list)

            for pred_file in pred_files:
                try:
                    dfp = pd.read_csv(pred_file, sep='\t', dtype=str)
                    for _, row in dfp.iterrows():
                        virus_id = str(row[0]).split('.')[-2]
                        if virus_id in EXCLUDED_VIRUSES:   # <-- add this
                            continue                        # <-- and this

                        state = row['State'].strip()
                        prob = float(row['Prediction'])
                        
                        virus_predictions[virus_id].append({
                            "state": state,
                            "prob": prob
                        })
                except Exception as e:
                    print(f"Error reading {pred_file}: {e}")
                    continue

            for virus_id, states in virus_predictions.items():
                sensitive_count = sum(1 for s in states if s["state"] == "sensitive")
                resistant_count = sum(1 for s in states if s["state"] == "resistant")
                total_votes = len(states)
                
                # Average probability of resistant across folds
                avg_prob_resistant = sum(s["prob"] for s in states) / total_votes

                if sensitive_count > resistant_count:
                    majority_label = 'sensitive'
                    confidence = sensitive_count / total_votes
                elif resistant_count > sensitive_count:
                    majority_label = 'resistant'
                    confidence = resistant_count / total_votes
                else:
                    majority_label = 'sensitive/resistant'
                    confidence = 0.5


                pred_frames.append({
                    'Antibody': antibody,
                    'Virus': virus_id,
                    'Threshold': float(thr),
                    'Predicted Label': majority_label,
                    'Sensitive_Votes': sensitive_count,
                    'Resistant_Votes': resistant_count,
                    'Total_Votes': total_votes,
                    'Confidence': round(confidence, 3),
                    'Prob_resistant': avg_prob_resistant
                })
        if not pred_frames:
            print(f" No predictions found for {antibody}, skipping...")
            continue

        pred_df = pd.concat([pd.DataFrame([p]) for p in pred_frames], ignore_index=True)
        print(f"Total predictions collected for {antibody}: {len(pred_df)}")

        # Merge with GT
        print("Merging predictions with ground truth data...")
        merged = pred_df.merge(
            geo_df,
            on=['Antibody','Virus','Threshold'],
            how='left'
        )
        merged['GeoMean_IC50'] = merged['GeoMean_IC50'].fillna('N/A')
        merged['GT Label'] = merged['GT Label'].fillna('N/A')

        print(f"After merging with GT: {len(merged)} rows (all predictions)")
        print(f"Predictions with GT data: {len(merged[merged['GT Label'] != 'N/A'])}")
        print(f"Predictions without GT data: {len(merged[merged['GT Label'] == 'N/A'])}")

        # Add clade
        print("Adding clade information...")
        merged_with_clade = merged.merge(
            viruses_clade_df,
            on='Virus',
            how='left'
        )

        viruses_with_clade = merged_with_clade[merged_with_clade['Clade'].notna()]['Virus'].nunique()
        viruses_total = merged_with_clade['Virus'].nunique()
        print(f"Clade information found for {viruses_with_clade} out of {viruses_total} unique viruses")

        all_pred_vs_gt_data.append(merged_with_clade)

        # Metrics (GT only)
        merged_with_gt = merged_with_clade[merged_with_clade['GT Label'] != 'N/A'].copy()
        print(f"Instances available for metrics calculation: {len(merged_with_gt)}")

        merged_with_gt['GT_bin'] = merged_with_gt['GT Label'].map({'sensitive': 0, 'resistant': 1}).astype(int)
        merged_with_gt['Pred_bin'] = merged_with_gt['Predicted Label'].map({'sensitive': 0, 'resistant': 1}).astype(int)

        records = []
        for (ab, thr), group in merged_with_gt.groupby(['Antibody','Threshold']):
            gt = group['GT_bin'].values
            pred = group['Pred_bin'].values
            prob = group['Prob_resistant'].values
            if len(gt) == 0:
                continue

            acc = sk.accuracy_score(gt, pred)
            mcc = sk.matthews_corrcoef(gt, pred)
            try:
                auc = sk.roc_auc_score(gt, prob)
            except ValueError:
                auc = float('nan')
            f1 = sk.f1_score(gt, pred, zero_division=0)
            # NOTE: keep original per-ab output columns; add Scenario only so we can compare
            records.append({
                'Scenario': scenario_name,
                'Antibody': ab,
                'Threshold': thr,
                'MCC': mcc,
                'AUC': auc,
                'ACC': acc,
                'F1': f1  
            })

        if records:
            metrics_df = pd.DataFrame(records)
            all_metrics_data.append(metrics_df)

        # Summary stats per antibody (includes Instances_with_GT)
        if len(merged_with_clade) > 0:
            total_instances = len(merged_with_clade)
            instances_with_gt = len(merged_with_gt)
            instances_without_gt = total_instances - instances_with_gt

            agreement_count = 0
            disagreement_count = 0
            tie_count = 0

            if instances_with_gt > 0:
                agreement_count = len(merged_with_gt[merged_with_gt['Predicted Label'] == merged_with_gt['GT Label']])
                disagreement_count = len(merged_with_gt[merged_with_gt['Predicted Label'] != merged_with_gt['GT Label']])
                tie_count = len(merged_with_gt[merged_with_gt['Predicted Label'] == 'sensitive/resistant'])

            avg_confidence = 0
            if len(merged_with_gt) > 0:
                avg_confidence = merged_with_gt['Confidence'].mean()

            summary_stats.append({
                'Scenario': scenario_name,
                'Antibody': antibody,
                'Total_Instances': total_instances,
                'Instances_with_GT': instances_with_gt,
                'Instances_without_GT': instances_without_gt,
                'Agreements': agreement_count,
                'Disagreements': disagreement_count,
                'Ties': tie_count,
                'Avg_Confidence': round(avg_confidence, 3),
                'Viruses_with_Clade': viruses_with_clade,
                'Total_Viruses': viruses_total
            })

        print(f"Completed processing {antibody}")

    combined_pred_vs_gt = pd.concat(all_pred_vs_gt_data, ignore_index=True) if all_pred_vs_gt_data else pd.DataFrame()
    combined_metrics = pd.concat(all_metrics_data, ignore_index=True) if all_metrics_data else pd.DataFrame()
    summary_df = pd.DataFrame(summary_stats) if summary_stats else pd.DataFrame()

    scenario_results[scenario_name] = {
        "pred_vs_gt": combined_pred_vs_gt,
        "metrics": combined_metrics,
        "summary": summary_df,
    }

    # Collect into combined summary tab
    if not combined_metrics.empty:
        all_scenarios_metrics.append(combined_metrics)
    if not summary_df.empty:
        all_scenarios_summary.append(summary_df)

# Build combined per-antibody metrics/summary across scenarios
combined_metrics_all = pd.concat(all_scenarios_metrics, ignore_index=True) if all_scenarios_metrics else pd.DataFrame()
combined_summary_all = pd.concat(all_scenarios_summary, ignore_index=True) if all_scenarios_summary else pd.DataFrame()

# ------------------------------------------------------------------------------
# 4) Write Excel: 3 scenario tabs (pred_vs_gt) + 1 summary tab (per-ab metrics + summary)
# ------------------------------------------------------------------------------
print(f"\n{'='*60}")
print("Saving all scenarios into one Excel (4 tabs)...")
print(f"{'='*60}")

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    # Tab 1-3: pred_vs_gt per scenario
    for scenario_name, data in scenario_results.items():
        sheet_name = scenario_name[:31]
        if not data["pred_vs_gt"].empty:
            data["pred_vs_gt"].to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            pd.DataFrame({"note": ["No pred_vs_gt data produced for this scenario."]}).to_excel(
                writer, sheet_name=sheet_name, index=False
            )

    # Tab 4: per-antibody metrics + summary (NO scenario-level averages)
    summary_sheet = "per_ab_metrics_summary"
    startrow = 0

    if not combined_metrics_all.empty:
        combined_metrics_all.to_excel(writer, sheet_name=summary_sheet, index=False, startrow=startrow)
        startrow += len(combined_metrics_all) + 3
    else:
        pd.DataFrame({"note": ["No metrics data produced."]}).to_excel(
            writer, sheet_name=summary_sheet, index=False, startrow=startrow
        )
        startrow += 5

    if not combined_summary_all.empty:
        combined_summary_all.to_excel(writer, sheet_name=summary_sheet, index=False, startrow=startrow)
    else:
        pd.DataFrame({"note": ["No summary statistics produced."]}).to_excel(
            writer, sheet_name=summary_sheet, index=False, startrow=startrow
        )

print(f"\n Saved scenario comparison Excel to: {output_xlsx}")

Loading viruses file for clade information...
Loaded clade information for 2858 viruses

SCENARIO: include_TBDs

Processing antibody: PGT121
  Missing prediction directory: /home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/prediction_results/CATNAP_399_strains/curated_data_epitope_with_all_lineage/threshold_0.2/include_TBDs_GTlabels_flipped/5folds_1_repeat
  Missing prediction directory: /home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/prediction_results/CATNAP_399_strains/curated_data_epitope_with_all_lineage/threshold_1/include_TBDs_GTlabels_flipped/5folds_1_repeat
  Found only 0 files for PGT121 threshold 50, expected 5
 No predictions found for PGT121, skipping...

Processing antibody: VRC01
  Missing prediction directory: /home/yujieq/work/ML_training/deep_hiv_ab_pred/final_model_predictions/CATNAP_strains/prediction_results/CATNAP_399_strains/curated_data_epitope_with_all_lineage/threshold_0.2/include_TBDs_G